<img src="media/Banner_waterlevel+esalogo.svg" width="100%" alt="Banner for EO AFRICA course" />


*R. Rietbroek, May 2026*

<font color=#cf0072>

# Water Balance tutorial for the larger Rufiji Basin
</font>

<font color=#cf0072>
    
## Introduction and goal of the tutorial
Simplified, the water balance of an entire catchment can be approximated with a bucket model which fullfils the water balance equation:

$$\frac{d TWS}{dt}= P-ET - R$$

Which describes how the change in terrestrial water storage over the entire catchment, $\frac{d TWS}{dt}$ is fed by precipitation, $P$, and decreases through losses by evapotranspiration, $ET$ and (river) runoff. In this formulation we have ignored the contributions from subsurface groundwater, and non-river surface runoff for simplicity.  The above equation is provided in terms of mass per time unit, but it can also expressed in a volumetric unit, when prescribing the density of water, in e.g. $m^{3}/s$, which is commonly used to express river discharge.

The water balance equation allows the retrieval of one component from the 2 other components. However, since components are never perfectly known, the errors in the inferred component are a combination of the errors from the 2 input components.

# Goal
In this assignment, you will use GRACE and GRACE follow-on data to compute a time series of $\frac{d TWS}{dt}$ averaged over the Rufiji catchment. Using a $P-ET$ estimate from ERA5 data, and estimate of  $R$ at the mouth of the rivers in the Catchment will be obtained.

The inferred discharge will be compared with water level variations from radar altimetry, and discharge from the [GEOGLOWS model](https://www.geoglows.org/).  You will be asked to reflect on the results and discuss the possible error sources.

</font>

<font color=#cf0072>

## Step 1 (~500 words): Introduction
1. Write a short introduction on the purpose of this notebook aimed at fellow students. Make sure to also describe the characteristics of the larger Rufiji catchment, and it's socio-economic and ecological roles.
2. Provide a flow chart of its working

### Hints & Tips
* You can create a flowchart with an external program and embed it in the markdown cell below

</font>

.. YOUR ANSWER ...

<font color=#cf0072>

## Step 2: Load the necessary modules

### Hints & Tips
* Are all modules, needed for this exercise, present? (i.e. you can restart and rerun the entire exercise in the specified order to check)
* Besides the standard modules (os etc.) do you motivate why some special modules are needed?

</font>


<font color=#cf0072>

## Step 3: Load the catchment polygon, and convert it to a mask expressed in spherical harmonical basin coefficients

### Hints and tips
* Use a [geopandas.Geodataframe](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.html) to read the `basin` layer from the polygon file: `Zambezi_geometry.gpkg`
* The above file holds multiple catchments, make sure to select the one related to the Zambezi
* [This example notebook](https://shxarray.wobbly.earth/latest/notebooks/Geometry2sphericalHarmonics.html) contains example code to convert a polygon to a set of spherical harmonic basin coefficients (i.e. a spectral representation of the basin which we will use later to average GRACE data.
* In particular the function `xr.DataArray.sh.from_geoseries` can be used for this (thus function internally uses [shxarray's polygon2sh function](https://shxarray.wobbly.earth/latest/references/shxarray.geom.polygons.html#shxarray.geom.polygons.polygon2sh)).
* Use a maximum degree which is consistent with the GRACE data (`nmax=96`)
* Also create a variable which holds the area of the Zambezi basin in [m<sup>2</sup>] (you need it later)
</font>


<font color=#cf0072>

## Step 4: Load the GRACE and GRACE-FO solutions, and a static gravity field (GOCO06S) catchment polygon, and convert it to terrestrial water storage

### Hints and tips
* Data with time variable gravity Stokes coefficients are listed in `monthly_n96`. You can use the [glob.glob](https://docs.python.org/3/library/glob.html), to generate a list of filenames. Make sure to check that the filenames are listed in chronological order and use a [sorting](https://docs.python.org/3/howto/sorting.html) if needed. Use [xarray.open_mfdataset](https://docs.xarray.dev/en/stable/generated/xarray.open_mfdataset.html) to open the list of files names and specify `engine='icgem'` as the loading engine.
* The static gravity field is provided in the file `GOCO06s.gfc.gz`.
* For debugging, you can use `display(ds_xxx)` to explore the different DataArray variables, coordinates and indexes in a dataset. The Stokes coefficients are found in the `xarray.Dataset` as `xarray.DataArray`'s named `cnm`.
* You can rebuild the spherical harmonic index, after e.g. reading coefficients from the files with `dsgrace=dsgrace.sh.build_nmindex()`
* Create anomalies of the GRACE/GRACE-FO coefficients by subtracting the static gravity. You can do this by truncating both datasets to maximum and minimum degrees `nmax=96, nmin=1` (i.e. apply the [`~~.sh.truncate(...)` accessor](https://shxarray.wobbly.earth/latest/references/shxarray.core.shxarbase.html#shxarray.core.shxarbase.ShXrBase.truncate) before subtracting.
* Converting the anomaly can be done using the [`~~.sh.tws(...)` accessor](https://shxarray.wobbly.earth/latest/references/shxarray.core.xr_accessor.html#shxarray.core.xr_accessor.SHDaAccessor.tws). You probably need to specify `ingravtype='stokes'` as an argument to tws to ensure that the input is interpreted as unitless Stokes coefficients
</font>


<font color=#cf0072>

## Step 5: Compute the basin averages of the terrestrial water storage

Basin averages expressed as a uniformlayer of equivalent water height can be computed entirely in the spectral domain (see e.g. [Swenson and Wahr 2002](https://agupubs.onlinelibrary.wiley.com/doi/full/10.1029/2001JB000576)).

$$\overline{TWS}(t)=\frac{1}{B_{00}}\sum_{n=0}^{n=nmax}\sum_{m=-n}^{m} B_{nm} \tilde{T}_{nm} (t)$$

Here $B_{nm}$ are the basin coefficients from step 3, and $\tilde{T}_{nm}$ are the (filtered) spherical harmonic coefficients of terrestrial water storage coefficients from step 4. This operation is implement in `shxarray` where the filter can be supplied on the fly. For example:

`dabasin_mean=datws.sh.basinav(dabasincoef,filter='DDK6')`

computes the basin average for all time points of the terrestrial water storage coefficients datws `DataArray` in the basin represented by the coefficients $B_{nm}$.


### Hints and tips
* You can try different filters (e.g. `filter='DDK5','DDK6','DDK7','Gauss400'`) and find one which does not damp the seasonal amplitude too much while keeping the time series not too noisy.
</font>


<font color=#cf0072>

## Step 6: Compute the  time derivatives of the terrestrial water storage, and visualize the computed TWS storages and flux

The basin averages from Step 5, represent storage anomalies. In order to derive the rate of change of the the TWS anomalies you still need to apply a numerical differentiation in time. A simple approximation is using [central differences](https://patrickwalls.github.io/mathematicalpython/differentiation/differentiation/):

$$\frac{d\overline{TWS}(t_{i})}{dt} \approx = \frac{1}{2}\left(\frac{\overline{TWS}(t_{i})-\overline{TWS}(t_{i-1})}{t_{i}-t_{i-1}}+\frac{\overline{TWS}(t_{i+1})-\overline{TWS}(t_{i})}{t_{i+1}-t_{i}}\right)$$ 

For the visualization part:
* In each plot, plot at least two filter versions to compare
* Annotate the figures with titles and axes with correct units and labels
* Use a legend to distinguish between lines

### Hints and tips
* Use the central differences implementation from [xarray](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.differentiate.html) (read the documentation and supply the correct arguments to the function!)
* Make sure to convert $\frac{d\overline{TWS}(t_{i})}{dt}$ (as uniform layer change in $m/s$ ) to volume $m^{3}/s$ using the area of the catchment
* You can use [subplots](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html) to create a 2-panel figure to plot both the storage and flux.
</font>


<font color=#cf0072>

## Step 7 (~500 words): Reflect on the found  GRACE/GRACE-FO time series of TWS and it's rate

In your discussion reflect on the following items:
* Explain how terrestrial water storage anomalies are linked to the gravity field
* Are the found values realistic (provide evidence from external sources)?
* What is the effect of filtering on the time series?
* What time variations can be found from the data (seasonal,inter-annual)? make quantifiable statements based on your data.


### Hints and tips
* Make quantifiable statements whereever possible
* In your answer provide the links to your online (reliable) sources.
</font>


... YOUR ANSWER ....

<font color=#cf0072>

## Step 8: Load P-ET ERA5 time series, create and inferred discharge estimate from the water balance equation
1. Load the P, ET data from `era5_prec_evap_flux_storage.nc` variables (P=`avg_tprate`,ET=`avg_ie`) and select the Zambezi time series.
2. Interpolate the P-ET tiem series on the dTWS /dt time series and infer a discharge estimate

</font>


In [ ]:
display(dspet)

<font color=#cf0072>

## Step 9 : Plot a time series of the water balance components
1. Appropriate label the x and y axes, i.e. specify units
</font>


<font color=#cf0072>

## Step 10 ( ~500 words ): Reflect on the found results, and discuss potential error sources in the input data and methods
In your reflection address the following points:
* Discuss the spatial and time resolution of the time series. To what extent are the (in)consistent?
* List potential error sources of the satellite gravimetry dTWS/dt time series
* List potential error sources of the satellite altimetry
* List error source in the P-ET time series
* Discuss the found discrepancies (offset,amplitudes, phases)

### Hints and tips
* Refer to the lecture slides
* Make your statements quantifiable where possible
</font>
